In [5]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("uciml/iris", output_dir="datasets")

import pandera as pa
from pandera.typing import DataFrame, Seriesprint("Path to dataset files:", path)

100%|██████████| 3.60k/3.60k [00:00<00:00, 4.31MB/s]

Extracting files...
Path to dataset files: datasets


In [16]:
import pandera.pandas as pa
from pandera.typing import Series
import pandas as pd

class IrisSchema(pa.DataFrameModel):
    """
    Esquema de validación para el dataset Iris (estructura real de Kaggle).
    """
    
    Id: Series[int] = pa.Field(
        gt=0,
        description="ID único del registro",
        nullable=False
    )
    
    SepalLengthCm: Series[float] = pa.Field(
        gt=0,
        le=10,
        description="Longitud del sépalo en cm",
        nullable=False
    )
    
    SepalWidthCm: Series[float] = pa.Field(
        gt=0,
        le=10,
        description="Ancho del sépalo en cm",
        nullable=False
    )
    
    PetalLengthCm: Series[float] = pa.Field(
        gt=0,
        le=10,
        description="Longitud del pétalo en cm",
        nullable=False
    )
    
    PetalWidthCm: Series[float] = pa.Field(
        gt=0,
        le=10,
        description="Ancho del pétalo en cm",
        nullable=False
    )
    
    Species: Series[str] = pa.Field(
        isin=['Iris-setosa', 'Iris-versicolor', 'Iris-virginica'],
        description="Especie de Iris",
        nullable=False
    )
    
    class Config:
        strict = True
        coerce = True
        ordered = False
    
    @pa.dataframe_check
    def check_no_duplicates(cls, df: pd.DataFrame) -> bool:
        """Verifica que no haya filas duplicadas"""
        return ~df.duplicated().any()
    
    @pa.dataframe_check
    def check_unique_ids(cls, df: pd.DataFrame) -> bool:
        """Verifica que los IDs sean únicos"""
        return df['Id'].is_unique


def validate_iris_data(df: pd.DataFrame) -> dict:
    """
    Valida el dataset Iris usando IrisSchema y genera estadísticas descriptivas.
    
    Args:
        df: DataFrame con datos de Iris (estructura Kaggle)
        
    Returns:
        dict con 'valid', 'errors', 'data' y 'stats'
    """
    result = {
        "valid": False,
        "errors": [],
        "data": None,
        "stats": {}
    }
    
    # Validar con el esquema
    try:
        validated_df = IrisSchema.validate(df, lazy=True)
        result["valid"] = True
        result["data"] = validated_df
        print("✓ Validación exitosa")
        
    except pa.errors.SchemaErrors as e:
        result["errors"] = e.failure_cases.to_dict('records')
        validated_df = df
        print(f"✗ {len(e.failure_cases)} errores encontrados")
    
    # Generar estadísticas
    print("\n" + "="*60)
    print("RESUMEN DE DATOS")
    print("="*60)
    
    # General
    print(f"\n📊 General:")
    print(f"  • Registros: {len(validated_df)}")
    print(f"  • Columnas: {len(validated_df.columns)}")
    print(f"  • Duplicados: {validated_df.duplicated().sum()}")
    print(f"  • Valores nulos: {validated_df.isnull().sum().sum()}")
    print(f"  • IDs únicos: {validated_df['Id'].is_unique}")
    
    result["stats"]["total_records"] = len(validated_df)
    result["stats"]["duplicates"] = validated_df.duplicated().sum()
    result["stats"]["missing_values"] = validated_df.isnull().sum().to_dict()
    result["stats"]["unique_ids"] = validated_df['Id'].is_unique
    
    # Distribución de especies
    if 'Species' in validated_df.columns:
        print(f"\n🏷️  Distribución por especie:")
        species_counts = validated_df['Species'].value_counts()
        result["stats"]["species_distribution"] = species_counts.to_dict()
        
        for species, count in species_counts.items():
            pct = count / len(validated_df) * 100
            print(f"  • {species}: {count} ({pct:.1f}%)")
    
    # Estadísticas numéricas (excluyendo Id)
    numeric_cols = [col for col in validated_df.select_dtypes(include=['float64', 'int64']).columns 
                    if col != 'Id']
    
    if len(numeric_cols) > 0:
        print(f"\n📈 Estadísticas numéricas:")
        print(f"{'Columna':<20} {'Media':<10} {'Std':<10} {'Min':<10} {'Max':<10}")
        print("-" * 60)
        
        result["stats"]["numeric"] = {}
        for col in numeric_cols:
            mean = validated_df[col].mean()
            std = validated_df[col].std()
            min_val = validated_df[col].min()
            max_val = validated_df[col].max()
            
            print(f"{col:<20} {mean:<10.2f} {std:<10.2f} {min_val:<10.2f} {max_val:<10.2f}")
            
            result["stats"]["numeric"][col] = {
                "mean": float(mean),
                "std": float(std),
                "min": float(min_val),
                "max": float(max_val),
                "median": float(validated_df[col].median())
            }
    
    # Correlaciones (sin incluir Id)
    feature_cols = ['SepalLengthCm', 'SepalWidthCm', 'PetalLengthCm', 'PetalWidthCm']
    if all(col in validated_df.columns for col in feature_cols):
        print(f"\n🔗 Correlaciones fuertes (|r| > 0.7):")
        corr_matrix = validated_df[feature_cols].corr()
        result["stats"]["correlations"] = corr_matrix.to_dict()
        
        found = False
        for i in range(len(feature_cols)):
            for j in range(i+1, len(feature_cols)):
                corr = corr_matrix.iloc[i, j]
                if abs(corr) > 0.7:
                    print(f"  • {feature_cols[i]} ↔ {feature_cols[j]}: {corr:.3f}")
                    found = True
        
        if not found:
            print("  • No se encontraron correlaciones fuertes")
    
    print("\n" + "="*60 + "\n")
    
    return result

In [17]:
# Cargar datos (sin necesidad de renombrar columnas)
df = pd.read_csv("datasets/Iris.csv")

# Validar directamente
result = validate_iris_data(df)

# Usar datos validados
if result["valid"]:
    validated_df = result["data"]
    print("✅ Datos listos para procesamiento")
    
    # Opcional: remover columna Id si no la necesitas
    # df_clean = validated_df.drop('Id', axis=1)
else:
    print("❌ Errores encontrados:")
    for error in result["errors"][:5]:
        print(f"  - Columna: {error.get('column')}, Check: {error.get('check')}")

✓ Validación exitosa

RESUMEN DE DATOS

📊 General:
  • Registros: 150
  • Columnas: 6
  • Duplicados: 0
  • Valores nulos: 0
  • IDs únicos: True

🏷️  Distribución por especie:
  • Iris-setosa: 50 (33.3%)
  • Iris-versicolor: 50 (33.3%)
  • Iris-virginica: 50 (33.3%)

📈 Estadísticas numéricas:
Columna              Media      Std        Min        Max       
------------------------------------------------------------
SepalLengthCm        5.84       0.83       4.30       7.90      
SepalWidthCm         3.05       0.43       2.00       4.40      
PetalLengthCm        3.76       1.76       1.00       6.90      
PetalWidthCm         1.20       0.76       0.10       2.50      

🔗 Correlaciones fuertes (|r| > 0.7):
  • SepalLengthCm ↔ PetalLengthCm: 0.872
  • SepalLengthCm ↔ PetalWidthCm: 0.818
  • PetalLengthCm ↔ PetalWidthCm: 0.963


✅ Datos listos para procesamiento


In [19]:
datos_con_errores = {
    'Id': [1, 2, 3, 3, 5, 6, 7, 8, 9, 10],  # ❌ ID duplicado (3)
    'SepalLengthCm': [5.1, -2.3, 6.4, 7.2, 11.5, 5.8, 6.3, 5.1, 0, 5.9],  # ❌ Valor negativo, fuera de rango, cero
    'SepalWidthCm': [3.5, 3.0, 3.2, 3.6, 2.9, None, 3.3, 2.5, 4.1, 3.8],  # ❌ Valor nulo
    'PetalLengthCm': [1.4, 1.3, 4.5, 4.7, 5.2, 4.1, 15.0, 1.5, 3.8, -1.2],  # ❌ Fuera de rango, negativo
    'PetalWidthCm': [0.2, 0.2, 1.5, 1.4, 2.0, 1.3, 2.5, 0.3, 1.2, 1.8],
    'Species': [
        'Iris-setosa',
        'Iris-versicolor', 
        'iris-virginica',  # ❌ Minúscula (debería ser Iris-virginica)
        'Iris-virginica',
        'Iris-unknown',  # ❌ Especie inválida
        None,  # ❌ Valor nulo
        'Iris-setosa',
        'Rosa',  # ❌ Especie totalmente incorrecta
        'Iris-versicolor',
        'Iris-virginica'
    ],
    'ExtraColumn': [1, 2, 3, 4, 5, 6, 7, 8, 9, 10]  # ❌ Columna no permitida (strict=True)
}
df_malo = pd.DataFrame(datos_con_errores)

# Validar directamente
result = validate_iris_data(df_malo)

# Usar datos validados
if result["valid"]:
    validated_df = result["data"]
    print("✅ Datos listos para procesamiento")
    
    # Opcional: remover columna Id si no la necesitas
    # df_clean = validated_df.drop('Id', axis=1)
else:
    print("❌ Errores encontrados:")
    for error in result["errors"][:5]:
        print(f"  - Columna: {error.get('column')}, Check: {error.get('check')}")

✗ 12 errores encontrados

RESUMEN DE DATOS

📊 General:
  • Registros: 10
  • Columnas: 7
  • Duplicados: 0
  • Valores nulos: 2
  • IDs únicos: False

🏷️  Distribución por especie:
  • Iris-setosa: 2 (20.0%)
  • Iris-versicolor: 2 (20.0%)
  • Iris-virginica: 2 (20.0%)
  • iris-virginica: 1 (10.0%)
  • Iris-unknown: 1 (10.0%)
  • Rosa: 1 (10.0%)

📈 Estadísticas numéricas:
Columna              Media      Std        Min        Max       
------------------------------------------------------------
SepalLengthCm        5.10       3.81       -2.30      11.50     
SepalWidthCm         3.32       0.49       2.50       4.10      
PetalLengthCm        4.03       4.35       -1.20      15.00     
PetalWidthCm         1.24       0.79       0.20       2.50      
ExtraColumn          5.50       3.03       1.00       10.00     

🔗 Correlaciones fuertes (|r| > 0.7):
  • No se encontraron correlaciones fuertes


❌ Errores encontrados:
  - Columna: IrisSchema, Check: column_in_schema
  - Columna: IrisSc

In [26]:
df = pd.read_parquet("iris_features.parquet")

In [27]:
df

,SepalLengthCm,SepalWidthCm,PetalLengthCm,PetalWidthCm,Species,processed_at
0,-0.900681,1.032057,-1.341272,-1.312977,0,2026-04-23 04:53:32.698681+00:00
1,-1.143017,-0.124958,-1.341272,-1.312977,0,2026-04-23 04:53:32.698681+00:00
2,-1.385353,0.337848,-1.398138,-1.312977,0,2026-04-23 04:53:32.698681+00:00
3,-1.506521,0.106445,-1.284407,-1.312977,0,2026-04-23 04:53:32.698681+00:00
4,-1.021849,1.263460,-1.341272,-1.312977,0,2026-04-23 04:53:32.698681+00:00
...,...,...,...,...,...,...
145,1.038005,-0.124958,0.819624,1.447956,2,2026-04-23 04:53:32.698681+00:00
146,0.553333,-1.281972,0.705893,0.922064,2,2026-04-23 04:53:32.698681+00:00
147,0.795669,-0.124958,0.819624,1.053537,2,2026-04-23 04:53:32.698681+00:00
148,0.432165,0.800654,0.933356,1.447956,2,2026-04-23 04:53:32.698681+00:00
